In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
FILE_PATH = "Data_2025_2026_Combined.xlsx"

In [3]:
BORDER = "=" * 70
SEP    = "-" * 70
 
def header(title):
    print(f"\n{BORDER}")
    print(f"  {title}")
    print(BORDER)
 
def sub(title):
    print(f"\n{SEP}")
    print(f"  {title}")
    print(SEP)
 

In [4]:
header("ĐỌC FILE DỮ LIỆU")
all_sheets: dict[str, pd.DataFrame] = pd.read_excel(FILE_PATH, sheet_name=None)
print(f"✅ Đọc thành công: {FILE_PATH}")
print(f"   Số sheet       : {len(all_sheets)}")
print(f"   Tên các sheet  : {list(all_sheets.keys())}")


  ĐỌC FILE DỮ LIỆU
✅ Đọc thành công: Data_2025_2026_Combined.xlsx
   Số sheet       : 12
   Tên các sheet  : ['Thức uống', 'Books', 'Clothing', 'Cosmetics', 'Home & Office', 'Food', 'Pet', 'Service', 'Medical', 'Electronics', 'Accessories', 'Jewelry']


In [5]:
header("SECTION 1: KIỂM TRA TRƯỜNG DỮ LIỆU")
 
EXPECTED_COLS = ["Category", "User ID", "Product", "Tên SP"]
 
# Mapping ý nghĩa từng trường
FIELD_MEANING = {
    "Category" : "Danh mục sản phẩm cấp cao (tiếng Anh, khớp tên sheet)",
    "User ID"  : "Mã định danh người dùng (hex 24 ký tự – MongoDB ObjectId)",
    "Product"  : "Nhóm sản phẩm con trong danh mục",
    "Tên SP"   : "Tên sản phẩm cụ thể trên hóa đơn (free-text)",
    "Tên SP "  : "⚠️  CỘT THỪA – 'Tên SP' có khoảng trắng cuối (trailing space). "
                 "Chỉ xuất hiện ở sheet 'Thức uống', cần hợp nhất với 'Tên SP'.",
}
 
print("\n📋 Ý nghĩa dự kiến từng trường:")
for col, desc in FIELD_MEANING.items():
    print(f"   {col:<12}: {desc}")
 
print("\n📊 Cột thực tế theo từng sheet:")
issues = []
for sheet, df in all_sheets.items():
    cols = df.columns.tolist()
    extra = [c for c in cols if c not in EXPECTED_COLS]
    missing = [c for c in EXPECTED_COLS if c not in cols]
    status = "✅" if not extra and not missing else "⚠️ "
    note = ""
    if extra:
        note += f" | THỪA: {extra}"
        issues.append((sheet, "Cột thừa", extra))
    if missing:
        note += f" | THIẾU: {missing}"
        issues.append((sheet, "Cột thiếu", missing))
    print(f"   {status} [{sheet:<15}] {cols}{note}")
 
print("\n📌 Tổng hợp vấn đề cột:")
if issues:
    for sheet, kind, cols in issues:
        print(f"   ⚠️  [{sheet}] {kind}: {cols}")
else:
    print("   Không phát hiện vấn đề.")
 
# Kiểm tra định dạng User ID (MongoDB ObjectId = 24 hex chars)
sub("Kiểm tra định dạng User ID (chuẩn MongoDB ObjectId – 24 ký tự hex)")
uid_problems = {}
for sheet, df in all_sheets.items():
    if "User ID" not in df.columns:
        continue
    invalid = df["User ID"].dropna().astype(str).apply(
        lambda x: not (len(x) == 24 and all(c in "0123456789abcdef" for c in x.lower()))
    )
    bad_vals = df.loc[invalid[invalid].index, "User ID"].unique()
    if len(bad_vals):
        uid_problems[sheet] = bad_vals[:5]
 
if uid_problems:
    for sheet, vals in uid_problems.items():
        print(f"   ⚠️  [{sheet}] User ID không hợp lệ ({len(vals)} mẫu): {vals}")
else:
    print("   ✅ Tất cả User ID đều đúng định dạng 24 ký tự hex.")
 
# Kiểm tra Category có khớp tên sheet không
sub("Kiểm tra Category khớp tên sheet")
SHEET_CATEGORY_MAP = {
    "Thức uống"  : "Beverages",
    "Books"      : "Books",
    "Clothing"   : "Clothing",
    "Cosmetics"  : "Cosmetic",   # chú ý không có 's'
    "Home & Office": "Home & Office",
    "Food"       : "Food",
    "Pet"        : "Pets",
    "Service"    : "Dịch vụ",
    "Medical"    : "Medical",
    "Electronics": "Electronics",
    "Accessories": "Accessories",
    "Jewelry"    : "Jewelry",
}
for sheet, expected_cat in SHEET_CATEGORY_MAP.items():
    df = all_sheets[sheet]
    unique_cats = df["Category"].dropna().unique().tolist()
    match = expected_cat in unique_cats
    status = "✅" if match else "⚠️ "
    print(f"   {status} [{sheet:<15}] Mong đợi='{expected_cat}'  |  Thực tế={unique_cats}")


  SECTION 1: KIỂM TRA TRƯỜNG DỮ LIỆU

📋 Ý nghĩa dự kiến từng trường:
   Category    : Danh mục sản phẩm cấp cao (tiếng Anh, khớp tên sheet)
   User ID     : Mã định danh người dùng (hex 24 ký tự – MongoDB ObjectId)
   Product     : Nhóm sản phẩm con trong danh mục
   Tên SP      : Tên sản phẩm cụ thể trên hóa đơn (free-text)
   Tên SP      : ⚠️  CỘT THỪA – 'Tên SP' có khoảng trắng cuối (trailing space). Chỉ xuất hiện ở sheet 'Thức uống', cần hợp nhất với 'Tên SP'.

📊 Cột thực tế theo từng sheet:
   ⚠️  [Thức uống      ] ['Category', 'User ID', 'Product', 'Tên SP ', 'Tên SP'] | THỪA: ['Tên SP ']
   ⚠️  [Books          ] ['Category', 'User ID', 'Product', 'Tên SP ', 'Tên SP'] | THỪA: ['Tên SP ']
   ✅ [Clothing       ] ['Category', 'User ID', 'Product', 'Tên SP']
   ✅ [Cosmetics      ] ['Category', 'User ID', 'Product', 'Tên SP']
   ✅ [Home & Office  ] ['Category', 'User ID', 'Product', 'Tên SP']
   ✅ [Food           ] ['Category', 'User ID', 'Product', 'Tên SP']
   ✅ [Pet            ] [

In [6]:
header("SECTION 2: CẤU TRÚC & KÍCH THƯỚC DATASET")
 
# Gộp tất cả sheet thành 1 dataframe (dùng cột Tên SP chuẩn)
frames = []
for sheet, df in all_sheets.items():
    df = df.copy()
    df["_sheet"] = sheet
    # Hợp nhất cột 'Tên SP ' (trailing space) vào 'Tên SP' nếu có
    if "Tên SP " in df.columns and "Tên SP" in df.columns:
        df["Tên SP"] = df["Tên SP"].fillna(df["Tên SP "])
        df = df.drop(columns=["Tên SP "])
    # Nếu chỉ có 'Tên SP ' mà không có 'Tên SP'
    elif "Tên SP " in df.columns:
        df = df.rename(columns={"Tên SP ": "Tên SP"})
    frames.append(df)
 
combined = pd.concat(frames, ignore_index=True)
print(f"\n📐 Dataset tổng hợp (sau khi gộp tất cả sheet):")
print(f"   Số dòng (tổng) : {len(combined):,}")
print(f"   Số cột chuẩn   : {len(combined.columns)} — {combined.columns.tolist()}")
 
print("\n📊 Kích thước từng sheet:")
print(f"   {'Sheet':<18} {'Dòng':>8} {'Cột':>6} {'Kiểu dữ liệu'}")
print(f"   {'-'*18} {'-'*8} {'-'*6} {'-'*30}")
for sheet, df in all_sheets.items():
    dtypes = df.dtypes.apply(lambda x: str(x)).unique().tolist()
    print(f"   {sheet:<18} {len(df):>8,} {len(df.columns):>6}   {dtypes}")
 
sub("Kiểu dữ liệu các cột (dataset tổng hợp)")
print(combined.dtypes.to_string())
 
sub("Số lượng giá trị duy nhất")
for col in ["Category", "User ID", "Product", "Tên SP"]:
    if col in combined.columns:
        n = combined[col].nunique()
        print(f"   {col:<12}: {n:,} giá trị duy nhất")
 


  SECTION 2: CẤU TRÚC & KÍCH THƯỚC DATASET

📐 Dataset tổng hợp (sau khi gộp tất cả sheet):
   Số dòng (tổng) : 13,365
   Số cột chuẩn   : 5 — ['Category', 'User ID', 'Product', 'Tên SP', '_sheet']

📊 Kích thước từng sheet:
   Sheet                  Dòng    Cột Kiểu dữ liệu
   ------------------ -------- ------ ------------------------------
   Thức uống             7,746      5   ['object']
   Books                    89      5   ['object']
   Clothing                104      4   ['object']
   Cosmetics               151      4   ['object']
   Home & Office            61      4   ['object']
   Food                  4,505      4   ['object']
   Pet                      63      4   ['object']
   Service                 121      4   ['object']
   Medical                 471      4   ['object']
   Electronics              35      4   ['object']
   Accessories              17      4   ['object']
   Jewelry                   2      4   ['object']

-------------------------------------------

In [7]:
header("SECTION 3: DỮ LIỆU THIẾU & TRÙNG LẶP DO HỆ THỐNG")
 
# 3A – Missing values
sub("3A. Dữ liệu thiếu (Missing Values)")
print(f"\n   Tổng quan dataset tổng hợp ({len(combined):,} dòng):")
missing_summary = combined.drop(columns=["_sheet"]).isnull().sum()
missing_pct     = (missing_summary / len(combined) * 100).round(2)
miss_df = pd.DataFrame({
    "Số dòng thiếu": missing_summary,
    "Tỷ lệ (%)": missing_pct
})
print(miss_df.to_string())
 
print("\n   Chi tiết missing theo sheet (chỉ hiện sheet có missing):")
for sheet, df in all_sheets.items():
    df_check = df.copy()
    if "Tên SP " in df_check.columns and "Tên SP" in df_check.columns:
        df_check["Tên SP"] = df_check["Tên SP"].fillna(df_check["Tên SP "])
        df_check = df_check.drop(columns=["Tên SP "])
    elif "Tên SP " in df_check.columns:
        df_check = df_check.rename(columns={"Tên SP ": "Tên SP"})
 
    miss = df_check.isnull().sum()
    has_miss = miss[miss > 0]
    if len(has_miss):
        print(f"\n   [{sheet}]")
        for col, cnt in has_miss.items():
            pct = cnt / len(df_check) * 100
            print(f"      {col:<12}: {cnt:>5} dòng thiếu ({pct:.1f}%)")
 
# 3B – Duplicates do hệ thống (exact duplicates)
sub("3B. Trùng lặp do hệ thống (Exact Duplicates)")
 
print(f"\n   {'Sheet':<18} {'Tổng dòng':>10} {'Trùng chính xác':>16} {'Tỷ lệ (%)':>12}")
print(f"   {'-'*18} {'-'*10} {'-'*16} {'-'*12}")
total_sys_dups = 0
for sheet, df in all_sheets.items():
    df_check = df.copy()
    if "Tên SP " in df_check.columns and "Tên SP" in df_check.columns:
        df_check["Tên SP"] = df_check["Tên SP"].fillna(df_check["Tên SP "])
        df_check = df_check.drop(columns=["Tên SP "])
    elif "Tên SP " in df_check.columns:
        df_check = df_check.rename(columns={"Tên SP ": "Tên SP"})
 
    n_dup = df_check.duplicated().sum()
    total_sys_dups += n_dup
    pct = n_dup / len(df_check) * 100 if len(df_check) else 0
    flag = " ⚠️ " if n_dup > 0 else "  ✅"
    print(f"{flag} {sheet:<18} {len(df_check):>10,} {n_dup:>16,} {pct:>11.1f}%")
 
total_rows = len(combined)
print(f"\n   Tổng trùng chính xác toàn bộ: {total_sys_dups:,} / {total_rows:,} "
      f"({total_sys_dups/total_rows*100:.1f}%)")
 
# Xem mẫu trùng từ sheet có nhiều nhất
worst_sheet = max(
    all_sheets.keys(),
    key=lambda s: all_sheets[s].duplicated().sum()
)
sub(f"Mẫu dữ liệu trùng chính xác — sheet có nhiều nhất: [{worst_sheet}]")
df_worst = all_sheets[worst_sheet].copy()
if "Tên SP " in df_worst.columns and "Tên SP" in df_worst.columns:
    df_worst["Tên SP"] = df_worst["Tên SP"].fillna(df_worst["Tên SP "])
    df_worst = df_worst.drop(columns=["Tên SP "])
mask = df_worst.duplicated(keep=False)
sample = df_worst[mask].sort_values(df_worst.columns.tolist()).head(10)
print(sample.to_string(index=False))
 
 


  SECTION 3: DỮ LIỆU THIẾU & TRÙNG LẶP DO HỆ THỐNG

----------------------------------------------------------------------
  3A. Dữ liệu thiếu (Missing Values)
----------------------------------------------------------------------

   Tổng quan dataset tổng hợp (13,365 dòng):
          Số dòng thiếu  Tỷ lệ (%)
Category              4       0.03
User ID               2       0.01
Product               2       0.01
Tên SP                2       0.01

   Chi tiết missing theo sheet (chỉ hiện sheet có missing):

   [Cosmetics]
      Category    :     2 dòng thiếu (1.3%)

   [Home & Office]
      Category    :     2 dòng thiếu (3.3%)
      User ID     :     2 dòng thiếu (3.3%)
      Product     :     2 dòng thiếu (3.3%)
      Tên SP      :     2 dòng thiếu (3.3%)

----------------------------------------------------------------------
  3B. Trùng lặp do hệ thống (Exact Duplicates)
----------------------------------------------------------------------

   Sheet               Tổng dòng  Trùng

In [8]:
header("SECTION 4: PHÂN TÍCH TRÙNG LẶP DẠNG GIAN LẬN HÓA ĐƠN")
 
print("""
📌 Định nghĩa "gian lận hóa đơn" trong ngữ cảnh này:
   Một User ID khai báo cùng sản phẩm (Product + Tên SP) nhiều lần
   trong cùng 1 danh mục, với tần suất bất thường, có thể nhằm
   tính nhiều lần chi phí cho cùng 1 giao dịch.
 
   Các mẫu được kiểm tra:
   (A) Cùng User ID + Product + Tên SP lặp ≥ 3 lần trong 1 sheet
   (B) Cùng User ID xuất hiện ở nhiều sheet với cùng sản phẩm
   (C) User ID có số lần mua hàng bất thường cao so với phân vị 95%
""")
 
# Chuẩn bị combined không có exact dups
combined_clean = combined.drop_duplicates()
 
# ── 4A: Cùng user + product + tên SP lặp >= 3 lần trong 1 sheet ──────
sub("4A. User ID khai báo cùng sản phẩm ≥ 3 lần trong 1 sheet (sau loại exact dup)")
group_cols = ["_sheet", "Category", "User ID", "Product", "Tên SP"]
freq = (
    combined_clean.groupby(group_cols, dropna=False)
    .size()
    .reset_index(name="count")
)
suspicious_a = freq[freq["count"] >= 3].sort_values("count", ascending=False)
print(f"\n   Số bản ghi nghi vấn: {len(suspicious_a)}")
if len(suspicious_a):
    print(f"\n   Top 20 bản ghi tần suất cao nhất:")
    print(suspicious_a.head(20).to_string(index=False))
 
# ── 4B: Cùng User ID + Tên SP xuất hiện ở nhiều sheet khác nhau ──────
sub("4B. Cùng User ID + Tên SP xuất hiện ở nhiều sheet khác nhau")
cross = (
    combined_clean.groupby(["User ID", "Tên SP"], dropna=False)["_sheet"]
    .nunique()
    .reset_index(name="n_sheets")
)
suspicious_b = cross[cross["n_sheets"] >= 2].sort_values("n_sheets", ascending=False)
print(f"\n   Số cặp (User ID, Tên SP) xuất hiện đa danh mục: {len(suspicious_b)}")
if len(suspicious_b):
    print(f"\n   Top 15:")
    top_b = suspicious_b.head(15).copy()
    # Thêm danh sách sheet
    def get_sheets(row):
        mask = (
            (combined_clean["User ID"] == row["User ID"]) &
            (combined_clean["Tên SP"] == row["Tên SP"])
        )
        return ", ".join(sorted(combined_clean.loc[mask, "_sheet"].unique()))
    top_b["Sheets"] = top_b.apply(get_sheets, axis=1)
    print(top_b.to_string(index=False))
 
# ── 4C: User ID tần suất bất thường ─────────────────────────────────
sub("4C. User ID có tần suất giao dịch bất thường (> phân vị 95%)")
user_freq = combined_clean.groupby("User ID").size().reset_index(name="total_records")
p95 = user_freq["total_records"].quantile(0.95)
p99 = user_freq["total_records"].quantile(0.99)
print(f"\n   Phân vị 50% (median) : {user_freq['total_records'].median():.0f} bản ghi / user")
print(f"   Phân vị 95%          : {p95:.0f} bản ghi / user")
print(f"   Phân vị 99%          : {p99:.0f} bản ghi / user")
print(f"   Max                  : {user_freq['total_records'].max()} bản ghi / user")
 
outlier_users = user_freq[user_freq["total_records"] > p95].sort_values(
    "total_records", ascending=False
)
print(f"\n   User ID vượt p95 ({p95:.0f} bản ghi): {len(outlier_users)} user")
print(f"\n   Top 20 user có tần suất cao nhất:")
top20 = outlier_users.head(20).copy()
# Thêm danh sách sheet mua hàng
def user_sheets(uid):
    return ", ".join(sorted(combined_clean.loc[combined_clean["User ID"]==uid, "_sheet"].unique()))
top20["Sheets mua hàng"] = top20["User ID"].apply(user_sheets)
print(top20.to_string(index=False))
 
# ── 4D: Tóm tắt user đáng nghi nhất (xuất hiện cả 3 dấu hiệu) ───────
sub("4D. Tổng hợp — User ID đáng nghi nhất (kết hợp các dấu hiệu)")
risky_users = set()
 
if len(suspicious_a):
    risky_users.update(suspicious_a["User ID"].unique())
if len(suspicious_b):
    risky_users.update(suspicious_b["User ID"].unique())
risky_users.update(outlier_users["User ID"].unique())
 
print(f"\n   Tổng User ID có ít nhất 1 dấu hiệu đáng nghi: {len(risky_users)}")
 
# Bảng tổng hợp rủi ro
risk_rows = []
for uid in risky_users:
    u_data = combined_clean[combined_clean["User ID"] == uid]
    a_flag = len(suspicious_a[suspicious_a["User ID"] == uid]) > 0 if len(suspicious_a) else False
    b_flag = len(suspicious_b[suspicious_b["User ID"] == uid]) > 0 if len(suspicious_b) else False
    c_flag = uid in outlier_users["User ID"].values
    n_rec  = len(u_data)
    sheets = ", ".join(sorted(u_data["_sheet"].unique()))
    risk_rows.append({
        "User ID"       : uid,
        "Tổng bản ghi"  : n_rec,
        "A: Lặp ≥3"     : "✔" if a_flag else "",
        "B: Đa danh mục": "✔" if b_flag else "",
        "C: Bất thường" : "✔" if c_flag else "",
        "Sheets"        : sheets,
    })
 
risk_df = (
    pd.DataFrame(risk_rows)
    .sort_values("Tổng bản ghi", ascending=False)
    .reset_index(drop=True)
)
print("\n   (Hiển thị top 30 user rủi ro cao nhất theo số bản ghi)")
print(risk_df.head(30).to_string(index=False))
 


  SECTION 4: PHÂN TÍCH TRÙNG LẶP DẠNG GIAN LẬN HÓA ĐƠN

📌 Định nghĩa "gian lận hóa đơn" trong ngữ cảnh này:
   Một User ID khai báo cùng sản phẩm (Product + Tên SP) nhiều lần
   trong cùng 1 danh mục, với tần suất bất thường, có thể nhằm
   tính nhiều lần chi phí cho cùng 1 giao dịch.

   Các mẫu được kiểm tra:
   (A) Cùng User ID + Product + Tên SP lặp ≥ 3 lần trong 1 sheet
   (B) Cùng User ID xuất hiện ở nhiều sheet với cùng sản phẩm
   (C) User ID có số lần mua hàng bất thường cao so với phân vị 95%


----------------------------------------------------------------------
  4A. User ID khai báo cùng sản phẩm ≥ 3 lần trong 1 sheet (sau loại exact dup)
----------------------------------------------------------------------

   Số bản ghi nghi vấn: 0

----------------------------------------------------------------------
  4B. Cùng User ID + Tên SP xuất hiện ở nhiều sheet khác nhau
----------------------------------------------------------------------

   Số cặp (User ID, Tên SP) xuất h

In [9]:
header("SECTION 5: TÓM TẮT & KHUYẾN NGHỊ")
 
total_records     = len(combined)
total_clean       = len(combined_clean)
sys_dup_rate      = (total_records - total_clean) / total_records * 100
fraud_suspect_cnt = len(risky_users)
 
print(f"""
📊 TỔNG QUAN DỮ LIỆU
   ├─ Tổng số sheet       : {len(all_sheets)}
   ├─ Tổng số bản ghi gốc : {total_records:,}
   ├─ Tổng sau loại trùng : {total_clean:,}
   └─ Tỷ lệ trùng hệ thống: {sys_dup_rate:.1f}%
 
⚠️  VẤN ĐỀ CHẤT LƯỢNG DỮ LIỆU
   1. Cột "Tên SP " (có trailing space) ở sheet "Thức uống" bị tách riêng
      → Cần hợp nhất với cột "Tên SP" chuẩn.
   2. Missing values: Cột "Tên SP" có {combined['Tên SP'].isna().sum():,} giá trị null
      ({combined['Tên SP'].isna().sum()/len(combined)*100:.1f}%) — chủ yếu từ sheet Thức uống.
   3. Dữ liệu trùng chính xác chiếm {sys_dup_rate:.1f}% — cần xóa trước khi phân tích.
   4. Tên danh mục không nhất quán: "Cosmetic" (thiếu 's'), "Dịch vụ" (tiếng Việt)
      trong khi các sheet khác dùng tiếng Anh.
 
🔴 NGHI VẤN GIAN LẬN HÓA ĐƠN
   ├─ User đáng nghi (≥1 dấu hiệu): {fraud_suspect_cnt}
   ├─ Dấu hiệu A (lặp ≥3 lần / sheet): {len(suspicious_a)} cặp bản ghi
   ├─ Dấu hiệu B (đa danh mục, cùng SP): {len(suspicious_b)} cặp (User, Tên SP)
   └─ Dấu hiệu C (bất thường > P95={p95:.0f} bản ghi): {len(outlier_users)} user
 
✅ KHUYẾN NGHỊ
   1. Hợp nhất cột "Tên SP " → "Tên SP" và chuẩn hóa trailing space.
   2. Xóa exact duplicates trước mọi phân tích downstream.
   3. Điều tra {fraud_suspect_cnt} User ID có dấu hiệu gian lận — ưu tiên
      các user vừa lặp sản phẩm (A) vừa đa danh mục (B).
   4. Chuẩn hóa trường Category: thống nhất tiếng Anh, thêm 's' vào 'Cosmetic'.
   5. Bổ sung trường timestamp/ngày giao dịch để phân tích theo thời gian.
""")
 
print(f"\n{'='*70}")
print(f"  ✅ Phân tích hoàn tất.")
print(f"{'='*70}\n")


  SECTION 5: TÓM TẮT & KHUYẾN NGHỊ

📊 TỔNG QUAN DỮ LIỆU
   ├─ Tổng số sheet       : 12
   ├─ Tổng số bản ghi gốc : 13,365
   ├─ Tổng sau loại trùng : 10,007
   └─ Tỷ lệ trùng hệ thống: 25.1%

⚠️  VẤN ĐỀ CHẤT LƯỢNG DỮ LIỆU
   1. Cột "Tên SP " (có trailing space) ở sheet "Thức uống" bị tách riêng
      → Cần hợp nhất với cột "Tên SP" chuẩn.
   2. Missing values: Cột "Tên SP" có 2 giá trị null
      (0.0%) — chủ yếu từ sheet Thức uống.
   3. Dữ liệu trùng chính xác chiếm 25.1% — cần xóa trước khi phân tích.
   4. Tên danh mục không nhất quán: "Cosmetic" (thiếu 's'), "Dịch vụ" (tiếng Việt)
      trong khi các sheet khác dùng tiếng Anh.

🔴 NGHI VẤN GIAN LẬN HÓA ĐƠN
   ├─ User đáng nghi (≥1 dấu hiệu): 63
   ├─ Dấu hiệu A (lặp ≥3 lần / sheet): 0 cặp bản ghi
   ├─ Dấu hiệu B (đa danh mục, cùng SP): 35 cặp (User, Tên SP)
   └─ Dấu hiệu C (bất thường > P95=40 bản ghi): 50 user

✅ KHUYẾN NGHỊ
   1. Hợp nhất cột "Tên SP " → "Tên SP" và chuẩn hóa trailing space.
   2. Xóa exact duplicates trước mọ